In [56]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Carico il dataset pulito e sporco
df = pd.read_csv("datasetpronto.csv")
dfsporco = pd.read_csv("datasetsporcopronto.csv")

# Estraggo la lista dei testi (plot puliti e sporchi) e delle etichette (generi)
testi = df['Cleaned_Plot'].tolist()
testisporchi = dfsporco['Plot'].tolist()
etichette = df['Genre'].tolist()

# creo vettorizzatore tfidf e CountVectorizer senza rimuovere stopwords e poi con stopwords rimosse
TFIDFvectorizer = TfidfVectorizer()
CVvectorizer = CountVectorizer()

TFIDFvectorizer_no_stopwords = TfidfVectorizer(stop_words='english')
CVvectorizer_no_stopwords = CountVectorizer(stop_words='english')

In [57]:
# Vettorizzazione con TF-IDF con e senza stopwords


# Faccio la vettorizzazione su testi

X = TFIDFvectorizer.fit_transform(testi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {X.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {X.shape[1]}")
print()

# Versione SENZA le stopwords

X_no_stopwords = TFIDFvectorizer_no_stopwords.fit_transform(testi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {X_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {X_no_stopwords.shape[1]}")

Vettorizzazione CON stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57822

Vettorizzazione SENZA stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57519


Il numero di caratteristiche senza le stopword è minore perché semplicemente ha tolto 303 stopwords (303 parole in meno!)

In [ ]:
# Vettorizzazione con CountVectorizer con e senza stopwords

# Applico il vettorizzatore ai testi

cvX = CVvectorizer.fit_transform(testi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {cvX.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {cvX.shape[1]}")
print()

# Versione SENZA le stopwords

cvX_no_stopwords = CVvectorizer_no_stopwords.fit_transform(testi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {cvX_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {cvX_no_stopwords.shape[1]}")

In [ ]:
# Vettorizzazione con TF-IDF con e senza stopwords CON DATASET SPORCO


# Versione CON le stopwords

Xsporca = TFIDFvectorizer.fit_transform(testisporchi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {Xsporca.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {Xsporca.shape[1]}")
print()

# Versione SENZA le stopwords

Xsporca_no_stopwords = TFIDFvectorizer_no_stopwords.fit_transform(testisporchi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {Xsporca_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {Xsporca_no_stopwords.shape[1]}")

D'ORA IN POI INIZIA IL MODELLO.

In [ ]:
# importo GridSearchCV
from sklearn.model_selection import GridSearchCV


# importo classificatore SVC
from sklearn.svm import SVC

# istanzia il modello SVC con i parametri di default kernel=rbf, C=1
svc=SVC()


parameters = [
    {'C': [1, 10, 100], 'kernel': ['linear']},
    {'C': [1, 10, 100], 'kernel': ['rbf']},
    {'C': [1, 10, 100], 'kernel': ['poly']}
]


grid_search = GridSearchCV(estimator = svc,  
                           param_grid = parameters,
                           scoring = 'accuracy',
                           cv = 5,
                           verbose=1)




grid_search.fit(X_no_stopwords, etichette)

In [ ]:
# esamino il miglior modello


# miglior punteggio del GridSearchCV

print('GridSearch CV best score : {:.4f}\n\n'.format(grid_search.best_score_))


# stampa i parametri che hanno dato i risultati migliori

print('Parameters that give the best results :','\n\n', (grid_search.best_params_))


# stampa qual è il modello migliore

print('\n\nEstimator that was chosen by the search :','\n\n', (grid_search.best_estimator_))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)

In [ ]:
#Addestramento modello con matrice TF-IDF con stopwords



# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, etichette, test_size=0.2, random_state=42, stratify=etichette)



# Addestramento
best_model.fit(X_train, y_train)



# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Addestramento modello con matrice IF-IDF senza stopwords


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)



# Addestramento
best_model.fit(X_train, y_train)



# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# modello addestrato con la matrice CountVectorizer con stopwords


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(cvX, etichette, test_size=0.2, random_state=42, stratify=etichette)



# Addestramento
best_model.fit(X_train, y_train)



# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# modello adesstrato con la matrice CountVectorizer senza stopwords


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(cvX_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)


# Addestramento
best_model.fit(X_train, y_train)


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

D'ORA IN POI FACCIAMO TEST SUL MODELLO CON IL DATASET SPORCO.

In [ ]:
#Addestramento modello con matrice TF-IDF senza stopwords CON DATASET SPORCO (n.b.: perché riguardo al dataset sporco, vediamo solo senza stopwords)


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(Xsporca_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)


# Addestramento
best_model.fit(X_train, y_train)


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

DALLA NOSTRA RICERCA ABBIAMO VISTO CHE:
- L'ACCURACY CON TF-IDF VECTORIZER E' MAGGIORE RISPETTO A COUNTVECTORIZER.

TF-IDF
- L'ACCURACY CON TF-IDF VECTORIZER E' MAGGIORE QUANDO RIMUOVO LE STOPWORDS RISPETTO A QUANDO LE TENGO.

COUNTVECTORIZER
- L'ACCURACY CON COUNTVECTORIZER E' MAGGIORE QUANDO RIMUOVO LE STOPWORDS RISPETTO A QUANDO LE TENGO.

DATASET SPORCO
- TF-IDF / senza stopwords / DATASET PULITO          CONFRONTO CON          TF-IDF / senza stopwords / DATASET SPORCO
         Accuracy: 0.7238636363636364                                                Accuracy: 0.7170454545454545

Quindi, pulire il dataset porta ad una maggiore accuracy.




FUNZIONE CHE OTTIMIZZA IL PROCESSO DI ADDESTRAMENTO DEI MODELLI CREATI DA MATRICI DIVERSE

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

def addestra_e_valuta(X, y, nome_descrizione):

    print(f"Addestramento con: {nome_descrizione}")

    # Divisione train/test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Istanziazione e addestramento
    model = SVC(C=1, kernel='linear')
    model.fit(X_train, y_train)

    # Predizione
    y_pred = model.predict(X_test)

    # Valutazione
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    return model


In [52]:
# per far partire uno dei 4 modelli commento gli altri 3

# # TF-IDF con stopwords
# modello_tfidf_con = addestra_e_valuta(X, etichette, "TF-IDF con stopwords")

# # TF-IDF senza stopwords
modello_tfidf_senza = addestra_e_valuta(X_no_stopwords, etichette, "TF-IDF senza stopwords")

# # CountVectorizer con stopwords
# modello_cv_con = addestra_e_valuta(cvX, etichette, "CountVectorizer con stopwords")

# # CountVectorizer senza stopwords
# modello_cv_senza = addestra_e_valuta(cvX_no_stopwords, etichette, "CountVectorizer senza stopwords")

# DATASET SPORCO

# TF-IDF senza stopwords
# modello_tfidf_con_sporco = addestra_e_valuta(Xsporca_no_stopwords, etichette, "TF-IDF senza stopwords (dataset sporco)")


Addestramento con: TF-IDF senza stopwords
Accuracy: 0.7238636363636364

Classification Report:
               precision    recall  f1-score   support

      action       0.78      0.80      0.79       220
      comedy       0.64      0.69      0.66       220
       drama       0.62      0.62      0.62       220
      horror       0.86      0.79      0.82       220

    accuracy                           0.72       880
   macro avg       0.73      0.72      0.73       880
weighted avg       0.73      0.72      0.73       880



In [44]:
risultati = []

for nome, matrice in [('TF-IDF (con)', X), ('TF-IDF (senza)', X_no_stopwords),
                      ('CV (con)', cvX), ('CV (senza)', cvX_no_stopwords), ('Dataset sporco TF-IDF (senza)', Xsporca_no_stopwords)]:
    model = SVC(kernel='linear', C=1)
    X_train, X_test, y_train, y_test = train_test_split(matrice, etichette, test_size=0.2, stratify=etichette, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    risultati.append((nome, acc))

risultati


[('TF-IDF (con)', 0.7147727272727272),
 ('TF-IDF (senza)', 0.7238636363636364),
 ('CV (con)', 0.6136363636363636),
 ('CV (senza)', 0.6318181818181818),
 ('Dataset sporco TF-IDF (senza)', 0.7170454545454545)]

In [ ]:
#PROVE PER FEATURES 1/2

import numpy as np

# Recupera il vocabolario usato nel modello TF-IDF
feature_names = TFIDFvectorizer_no_stopwords.get_feature_names_out()

# Ottieni i coefficienti del classificatore lineare
# Ogni riga di coef_ corrisponde a una classe
classi = best_model.classes_  # ['action', 'comedy', 'drama', 'horror']
coefficienti = best_model.coef_

# Trova l'indice associato alla classe 'horror'
indice_horror = list(classi).index('horror')

# Ottieni i coefficienti per la classe 'horror'
coefficienti_horror = coefficienti[indice_horror].toarray().flatten()

# Ordina le features per importanza (in valore assoluto o positivo/negativo)
top_n = 30  # quante parole vuoi visualizzare
top_indici = np.argsort(coefficienti_horror)[-top_n:][::-1]

print(f"Top {top_n} parole più indicative per la classe 'horror':")
for indice in top_indici:
    print(f"{feature_names[indice]}: {coefficienti_horror[indice]:.4f}")

Top 30 parole più indicative per la classe 'drama':
kowai: 1.4820
fairton: 1.4601
monsignor: 1.3376
emu: 1.2280
layabout: 1.2226
commandeered: 1.1231
saits: 1.0880
jess: 1.0496
confuse: 1.0458
genuinely: 1.0142
allenby: 1.0090
lynskey: 1.0061
mcphillip: 0.9946
musskan: 0.9803
chandans: 0.9731
aires: 0.9708
newfoundland: 0.9696
soleil: 0.9620
heists: 0.9576
scrimshaw: 0.9492
july: 0.9471
koizumi: 0.9446
interlochen: 0.9395
rickman: 0.8992
lankford: 0.8885
kennels: 0.8806
goeto: 0.8529
saito: 0.8477
bloodhound: 0.8360
channards: 0.8292


In [65]:
#PROVE PER FEATURES 2/2

import numpy as np

# Recupera il vocabolario
feature_names = TFIDFvectorizer_no_stopwords.get_feature_names_out()

# Ottieni i coefficienti per tutte le classi
coefficienti = best_model.coef_.toarray()
classi = best_model.classes_

# Ottieni i pesi della classe horror
indice_horror = list(classi).index('horror')
pesi_horror = coefficienti[indice_horror]

# Calcola la media dei pesi delle ALTRE classi
pesi_altre_classi = np.mean(np.delete(coefficienti, indice_horror, axis=0), axis=0)

# Calcola la differenza dei pesi: horror vs altri
delta = pesi_horror - pesi_altre_classi


# Ordina in base alla differenza
top_n = 20
indici_top = np.argsort(delta)[-top_n:][::-1]

print(f"Top {top_n} parole distintive per 'horror' rispetto alle altre classi:")
for indice in indici_top:
    print(f"{feature_names[indice]}: delta={delta[indice]:.4f}, horror_weight={pesi_horror[indice]:.4f}, others_mean={pesi_altre_classi[indice]:.4f}")

Top 20 parole distintive per 'horror' rispetto alle altre classi:
goro: delta=1.7353, horror_weight=0.3028, others_mean=-1.4325
bernice: delta=1.6440, horror_weight=1.3918, others_mean=-0.2522
detect: delta=1.6314, horror_weight=0.0602, others_mean=-1.5712
gordies: delta=1.4517, horror_weight=1.3487, others_mean=-0.1030
househusband: delta=1.4489, horror_weight=1.0998, others_mean=-0.3491
jijoyand: delta=1.4396, horror_weight=1.2468, others_mean=-0.1928
foretells: delta=1.4336, horror_weight=1.0278, others_mean=-0.4058
feuding: delta=1.4313, horror_weight=1.1842, others_mean=-0.2470
aggravations: delta=1.3953, horror_weight=1.0021, others_mean=-0.3932
self: delta=1.3924, horror_weight=1.2056, others_mean=-0.1868
biz: delta=1.3726, horror_weight=0.2476, others_mean=-1.1251
ryūjis: delta=1.3320, horror_weight=1.1225, others_mean=-0.2094
everlys: delta=1.3249, horror_weight=1.0018, others_mean=-0.3230
chowkidar: delta=1.3213, horror_weight=1.2513, others_mean=-0.0699
feud: delta=1.3074, h

In [ ]:
import matplotlib.pyplot as plt
plt.barh(feature_names[indici_top], pesi_horror[indici_top], color='red', alpha=0.5, label='Horror')
plt.barh(feature_names[indici_top], pesi_altre_classi[indici_top], color='gray', alpha=0.5, label='Altre classi')
plt.legend()